# 00 Data audit (ASVspoof 2019 LA, train + dev only)

Class balance, duration and leading/trailing silence by class. Eval and In-the-Wild are not
touched here (they stay locked until `hypotheses.md` is frozen). Silence comes from the
energy VAD in `deepvoice.silence` (frame = speech if within 35 dB of the clip's loudest frame).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
proto = pd.read_parquet(ROOT / "data/protocols.parquet")
df = pd.concat(
    [pd.read_parquet(ROOT / f"data/features/silence_{s}.parquet").assign(split=s) for s in ["train", "dev"]]
).merge(proto[["file", "label", "attack", "speaker"]], on="file")
df["cls"] = df["label"].map({0: "real", 1: "fake"})

## Protocol counts (all splits; labels only, no audio)

In [ ]:
from deepvoice.data import summarize

counts = summarize(proto)
counts.to_csv(ROOT / "results/protocol_counts.csv", index=False)
counts

## Class balance

In [ ]:
bal = df.groupby(["split", "cls"], sort=False).size().unstack()
bal["fake_share"] = (bal["fake"] / bal.sum(axis=1)).round(3)
bal

## Duration and silence by class (median and interquartile range)

In [ ]:
rows = []
for (split, cls), g in df.groupby(["split", "cls"], sort=False):
    r = {"split": split, "class": cls, "n": len(g)}
    for c in ["duration", "lead_sil", "trail_sil", "sil_ratio", "energy_max"]:
        r[c + "_median"] = round(g[c].median(), 3)
        r[c + "_iqr"] = f"{g[c].quantile(.25):.2f}-{g[c].quantile(.75):.2f}"
    rows.append(r)
audit = pd.DataFrame(rows)
audit.to_csv(ROOT / "results/audit_by_class.csv", index=False)
audit.T

In [ ]:
df[df.split == "train"].groupby("attack")[["duration", "lead_sil", "trail_sil", "sil_ratio"]].median().round(3)

## Figure for the progress report

In [ ]:
REAL, FAKE = "#2a78d6", "#eb6834"  # categorical slots 1-2 (colour-blind checked)
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
fig, ax = plt.subplots(1, 3, figsize=(12, 3.6), gridspec_kw={"width_ratios": [1, 2.4, 1.3]})

# (a) class balance
for i, split in enumerate(["train", "dev"]):
    r, f = bal.loc[split, "real"], bal.loc[split, "fake"]
    ax[0].barh(i, r, color=REAL, height=0.6)
    ax[0].barh(i, f, left=r + 300, color=FAKE, height=0.6)
    ax[0].text(r + f + 900, i, f"{f / (r + f):.0%} fake", va="center", color=INK)
ax[0].set_yticks([0, 1], ["train", "dev"]); ax[0].invert_yaxis()
ax[0].set_xlim(0, 33000); ax[0].set_xlabel("clips")
ax[0].set_title("(a) Class balance", loc="left", color=INK)

# (b) leading silence: real vs each training attack, train and dev side by side
groups = ["bonafide", "A01", "A02", "A03", "A04", "A05", "A06"]
for j, split in enumerate(["train", "dev"]):
    d = df[df.split == split]
    data = [d.loc[d.attack == g, "lead_sil"] for g in groups]
    pos = [k * 3 + j for k in range(len(groups))]
    bp = ax[1].boxplot(data, positions=pos, widths=0.8, showfliers=False, patch_artist=True,
                       medianprops={"color": INK, "linewidth": 1.2})
    for patch, g in zip(bp["boxes"], groups):
        patch.set(facecolor=REAL if g == "bonafide" else FAKE, edgecolor="white",
                  alpha=1.0 if split == "train" else 0.55)
ax[1].set_xticks([k * 3 + 0.5 for k in range(len(groups))], ["real"] + groups[1:])
ax[1].set_ylabel("leading silence (s)"); ax[1].grid(axis="y", color=GRID, linewidth=0.6)
ax[1].set_title("(b) Leading silence by attack (left box train, right box dev)", loc="left", color=INK)

# (c) duration by class
for cls, col in [("real", REAL), ("fake", FAKE)]:
    ax[2].hist(df.loc[df.cls == cls, "duration"], bins=60, range=(0, 10), density=True,
               histtype="step", linewidth=2, color=col, label=cls)
ax[2].set_xlabel("duration (s)"); ax[2].set_yticks([])
ax[2].legend(frameon=False); ax[2].set_title("(c) Clip duration (train+dev)", loc="left", color=INK)

fig.tight_layout()
(ROOT / "results/figures").mkdir(exist_ok=True)
fig.savefig(ROOT / "results/figures/audit_silence.png", dpi=200)
fig.savefig(ROOT / "results/figures/audit_silence.pdf")